# POC: Interactive Documentation Assistant

This notebook demonstrates the complete end-to-end pipeline for generating project documentation on Google Colab T4 GPU.

## Features
- **Phase 1**: Static code analysis using tree-sitter (no LLM)
- **Phase 2**: Google-style docstring generation
- **Phase 3**: README generation (6 sections)
- **Phase 4**: Validation and iterative improvement
- **Phase 5**: Final quality evaluation

## Memory Optimization
- 4-bit quantization for T4 compatibility
- Small model (1.5B params)
- Content-based caching

## Installation

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torch sentencepiece bitsandbytes tree-sitter tree-sitter-languages networkx psutil pydantic

## Clone Repository (if needed)

In [ ]:
# Clone this repository if running on Colab
import os

if not os.path.exists('Multi-agent_Hierarchical_Documentation'):
    !git clone https://github.com/SalmaHisham/Multi-agent_Hierarchical_Documentation.git
    os.chdir('Multi-agent_Hierarchical_Documentation')
else:
    os.chdir('Multi-agent_Hierarchical_Documentation')

## Setup Environment

In [ ]:
import os
import sys

# Reduce HF/Transformers noise
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Add current directory to path
if '.' not in sys.path:
    sys.path.insert(0, '.')

## Import Modules

In [ ]:
from pipeline.orchestrator import Orchestrator
from pathlib import Path
import json

## Configure Project Path

Set the path to the project you want to document.

In [ ]:
# Example: document this repository itself
PROJECT_PATH = "."

# Or specify a different project
# PROJECT_PATH = "/path/to/your/project"

## Initialize Orchestrator

The orchestrator coordinates all 5 phases of the pipeline.

In [ ]:
orchestrator = Orchestrator(
    repo_path=PROJECT_PATH,
    artifacts_dir="./artifacts",
    model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",  # 1.5B for speed
    device="auto",  # Use GPU if available
    quantize=True,  # 4-bit quantization for T4
)

print(f"Project: {orchestrator.project_name}")
print(f"Path: {orchestrator.repo_path}")

## Phase 1: Static Analysis (No LLM)

Parse source code using tree-sitter to extract AST, dependencies, and components.

In [ ]:
phase1_results = orchestrator.run_phase1()

print(f"\nModules: {phase1_results['stats']['modules']}")
print(f"Functions: {phase1_results['stats']['functions']}")
print(f"Classes: {phase1_results['stats']['classes']}")

## Phase 2: Docstring Generation

Generate Google-style docstrings using lightweight LLM with caching.

In [ ]:
phase2_results = orchestrator.run_phase2()

print(f"\nTotal: {phase2_results['stats']['total']}")
print(f"Cached: {phase2_results['stats']['cached']}")
print(f"Generated: {phase2_results['stats']['generated']}")

## Phase 3: README Generation

Generate comprehensive README with 6 required sections.

In [ ]:
phase3_results = orchestrator.run_phase3()

print(f"\nREADME generated: {phase3_results['readme_path']}")

## Phase 4: Validation

Validate README sections using heuristics.

In [ ]:
phase4_results = orchestrator.run_phase4()

print(f"\nAll valid: {phase4_results['all_valid']}")
for section, result in phase4_results['sections'].items():
    status = "✅" if result['valid'] else "⚠️"
    print(f"{status} {section}")

## Phase 5: Evaluation

Evaluate documentation quality on 4 metrics.

In [ ]:
phase5_results = orchestrator.run_phase5()

evaluation = phase5_results['evaluation']
print(f"\nOverall Score: {evaluation['overall']:.1f}/10")
print(f"Clarity: {evaluation.get('clarity', 0)}")
print(f"Completeness: {evaluation.get('completeness', 0)}")
print(f"Consistency: {evaluation.get('consistency', 0)}")
print(f"Usability: {evaluation.get('usability', 0)}")

## View Generated README

In [ ]:
readme_path = Path(phase3_results['readme_path'])
if readme_path.exists():
    readme_content = readme_path.read_text()
    print(readme_content)
else:
    print("README not found")

## View Evaluation Report

In [ ]:
report_path = Path(phase5_results['report_path'])
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(json.dumps(report, indent=2))
else:
    print("Report not found")

## Cleanup

In [ ]:
# Free GPU memory
orchestrator.cleanup()
print("Cleanup complete!")

## Running All Phases at Once

Alternatively, you can run all phases with a single command:

In [ ]:
# Create a fresh orchestrator
orchestrator = Orchestrator(
    repo_path=PROJECT_PATH,
    artifacts_dir="./artifacts",
    model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    device="auto",
    quantize=True,
)

# Run all phases
all_results = orchestrator.run_all()

# Cleanup
orchestrator.cleanup()